# Qwen stitch selection + pattern recipes (free pilot)

This changes Qwen LoRA weights for a **new task**, rather than changing the earlier catalog adapter. The first scope is one flat swatch: peyote, brick, ladder, RAW or square stitch. A recipe specifies technique and dimensions; deterministic code produces the bead chart, thread operations and passage ledger.

**175 synthetic training examples; 22 development and 31 frozen challenge examples.** Positive recipe dimensions do not cross splits. Templates are synthetic and repetitive, so scores are not independent real-user validation. No physical-success labels or copied project patterns are used.

Use a free T4 GPU. Run setup, connect Drive if asked, then run training. No paid resources or quota bypass. Saves under `MyDrive/Beadform-free-qwen/runs/stitch-program-v1`, preserving earlier adapters. Both base and adapter predictions are compiled and compared. No automatic deployment.

These flat swatch routes exclude finishing and do not yet generate complete 3D subjects, spheres, tubes, mixed-stitch parts or arbitrary joints. The five-pass preference is checked for assembly; missing finishes stay explicitly flagged.


In [ ]:
import sys, subprocess, shutil, json
from pathlib import Path
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No free CUDA GPU. Stop and retry later; do not upgrade or bypass quotas.')
from google.colab import drive
drive.mount('/content/drive')
run=Path('/content/drive/MyDrive/Beadform-free-qwen/runs/stitch-program-v1')
run.mkdir(parents=True,exist_ok=True)
repo=Path('/content/beadform-stitch-program-v1')
if not repo.exists():
    subprocess.run(['git','clone','--branch','main','https://github.com/c-tustin/beadform.git',str(repo)],check=True)
if subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=repo,text=True).strip():
    raise RuntimeError('Source has edits; preserve them before proceeding.')
revision_file=run/'repository-revision.txt'
revision=revision_file.read_text().strip() if revision_file.exists() else subprocess.check_output(['git','rev-parse','HEAD'],cwd=repo,text=True).strip()
subprocess.run(['git','fetch','origin',revision],cwd=repo,check=True)
subprocess.run(['git','checkout','--detach',revision],cwd=repo,check=True)
revision_file.write_text(revision+chr(10))
if not shutil.which('node'):
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','-qq','nodejs'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(repo/'training/requirements-catalog-v2.txt')],check=True)
freeze=subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True)
p=run/'pip-freeze.txt'
if p.exists() and p.read_text()!=freeze:
    raise RuntimeError('Environment changed; inspect before resuming. Preserve the saved run.')
p.write_text(freeze)
print('Source',revision,'GPU',torch.cuda.get_device_name(0),flush=True)
subprocess.run([sys.executable,str(repo/'training/train_stitch_program.py'),'--output',str(run),'--check'],cwd=repo,check=True)


In [ ]:
command=[sys.executable,'-u',str(repo/'training/train_stitch_program.py'),'--output',str(run),'--resume']
if not (run/'comparison.json').exists():
    with (run/'run.log').open('a') as log:
        process=subprocess.Popen(command,cwd=repo,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        for line in process.stdout:
            log.write(line);log.flush();print(line,end='',flush=True)
        code=process.wait()
        if code:
            raise RuntimeError(f'Run stopped with code {code}. Inspect the saved log.')
print((run/'comparison.json').read_text())
print('Saved:',run)
print('Weights changed by QLoRA; no automatic deployment or physical validation.')
